# 胸部X光肺炎数据集 — 数据探索
本 Notebook 用于：
1. 查看数据集目录结构和样本数量
2. 可视化正常 vs 肺炎样本
3. 检查图像尺寸分布
4. 初步判断数据平衡情况

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, CLASS_NAMES

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('✅ 环境就绪')

In [ ]:
# 统计各数据集的样本数量
def count_images(path):
    normal = len(os.listdir(os.path.join(path, 'NORMAL'))) if os.path.isdir(os.path.join(path, 'NORMAL')) else 0
    pneumonia = len(os.listdir(os.path.join(path, 'PNEUMONIA'))) if os.path.isdir(os.path.join(path, 'PNEUMONIA')) else 0
    return normal, pneumonia

for split, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    n, p = count_images(path)
    print(f'{split:>6}: NORMAL={n:>5}, PNEUMONIA={p:>5}, Total={n+p:>5}')

In [ ]:
# 可视化：正常 vs 肺炎样本对比
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(cls_path):
        print(f'❌ {cls_path} 不存在，请先下载数据集')
        continue
    images = os.listdir(cls_path)[:4]
    for j, img_name in enumerate(images):
        img = Image.open(os.path.join(cls_path, img_name)).convert('RGB')
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].set_title(f'{cls} — {img.size[0]}×{img.size[1]}')
        axes[i, j].axis('off')

fig.suptitle('胸部X光样本对比：正常 vs 肺炎', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 检查图像尺寸分布
sizes = []
for cls in ['NORMAL', 'PNEUMONIA']:
    cls_path = os.path.join(TRAIN_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    for img_name in os.listdir(cls_path)[:500]:  # 抽样 500 张
        img = Image.open(os.path.join(cls_path, img_name))
        sizes.append(img.size)

if sizes:
    widths, heights = zip(*sizes)
    print(f'宽度范围: {min(widths)} ~ {max(widths)}, 均值: {np.mean(widths):.0f}')
    print(f'高度范围: {min(heights)} ~ {max(heights)}, 均值: {np.mean(heights):.0f}')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(widths, bins=30, edgecolor='black', alpha=0.7)
    ax1.set_title('图像宽度分布')
    ax2.hist(heights, bins=30, edgecolor='black', alpha=0.7)
    ax2.set_title('图像高度分布')
    plt.tight_layout()
    plt.show()